# ResumeGPT NER Fine-tuning (Phase 2)

Fine-tune existing NER model with remaining 13,922 examples.

**Prerequisites:**
- Phase 1 model trained (5,000 examples)
- Training data uploaded to Google Drive

**Expected Results:**
- F1 Score: 80-85% (up from 70-80%)
- Better entity detection for names, companies, skills

## Cell 1: Setup

In [ ]:
# Check GPU
!nvidia-smi

# Install spaCy
!pip install -q spacy

# Fix CuPy conflicts
!pip uninstall -y cupy-cuda11x cupy-cuda12x cupy 2>/dev/null
!pip install -q cupy-cuda12x

# Enable GPU
import spacy
if spacy.prefer_gpu():
    print('GPU enabled')
else:
    print('Using CPU')

print('Setup complete')

## Cell 2: Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set paths
DRIVE_PATH = '/content/drive/MyDrive/resumegpt_ner'
DATA_PATH = f'{DRIVE_PATH}/processed'
MODEL_OUTPUT = f'{DRIVE_PATH}/model_finetuned'

import os
import zipfile
os.makedirs(MODEL_OUTPUT, exist_ok=True)

# Check if data exists
if not os.path.exists(DATA_PATH):
    zip_path = f'{DRIVE_PATH}/processed.zip'
    if os.path.exists(zip_path):
        print('Extracting processed.zip...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(DRIVE_PATH)
        print('Extracted!')
    else:
        raise FileNotFoundError('Upload processed.zip to Drive first')

# List files
print('Files in data directory:')
for f in os.listdir(DATA_PATH):
    if f.endswith('.spacy') or f.endswith('.json'):
        size = os.path.getsize(f'{DATA_PATH}/{f}')
        print(f'  {f} ({size / 1024:.1f} KB)')

## Cell 3: Load Pre-trained Model

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
import random
import gc

# Enable GPU
spacy.prefer_gpu()

# Entity labels (same as Phase 1)
ENTITY_LABELS = [
    'PERSON', 'EMAIL', 'PHONE', 'LOCATION',
    'SKILL', 'COMPANY', 'JOB_TITLE',
    'DEGREE', 'SCHOOL', 'GRADUATION_YEAR',
    'CERTIFICATION', 'YEARS_EXPERIENCE',
    'LINKEDIN', 'URL', 'YEAR', 'GPA', 'GITHUB'
]

# Load Phase 1 model
print('Loading Phase 1 model...')
nlp = spacy.load('D:/resumegpt/backend/models/resume_ner')
print(f'Model loaded: {nlp.pipe_names}')
print(f'Labels: {list(nlp.pipe_labels.get("ner", []))}')

## Cell 4: Load Training Data

In [ ]:
# Load all training data
def load_spacy_data(file_path):
    doc_bin = DocBin().from_disk(file_path)
    return list(doc_bin.get_docs(nlp.vocab))

print('Loading training data...')
all_train_docs = load_spacy_data(f'{DATA_PATH}/train.spacy')
print(f'Total training examples: {len(all_train_docs)}')

# Skip first 5,000 (already used in Phase 1)
START_INDEX = 5000
additional_docs = all_train_docs[START_INDEX:]
print(f'Additional examples for fine-tuning: {len(additional_docs)}')

## Cell 5: Fine-tune Model

In [ ]:
# Fine-tuning config
N_EPOCHS = 15
BATCH_SIZE = 32
DROPOUT = 0.2
LEARNING_RATE = 0.0005  # Lower than Phase 1

print('=' * 60)
print('Fine-tuning NER Model')
print(f'Epochs: {N_EPOCHS}, Batch: {BATCH_SIZE}')
print('=' * 60)

# Create training examples
train_examples = []
for doc in additional_docs:
    pred_doc = nlp.make_doc(doc.text)
    example = Example(pred_doc, doc)
    train_examples.append(example)

print(f'Created {len(train_examples)} training examples')

# Fine-tune
import time
start_time = time.time()
optimizer = nlp.resume_training()
optimizer.learn_rate = LEARNING_RATE

for epoch in range(N_EPOCHS):
    epoch_start = time.time()
    random.shuffle(train_examples)
    losses = {}
    
    batches = spacy.util.minibatch(train_examples, size=BATCH_SIZE)
    for batch in batches:
        nlp.update(batch, drop=DROPOUT, losses=losses)
    
    epoch_time = time.time() - epoch_start
    loss = losses.get('ner', 0)
    print(f'Epoch {epoch+1}/{N_EPOCHS} - Loss: {loss:.4f} - Time: {epoch_time:.1f}s')
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        nlp.to_disk(f'{MODEL_OUTPUT}/model-finetuned-{epoch+1}')
        print(f'  -> Checkpoint saved')

# Save final model
nlp.to_disk(f'{MODEL_OUTPUT}/model-finetuned-final')
total_time = time.time() - start_time

print('\n' + '=' * 60)
print(f'Fine-tuning complete! Total time: {total_time/60:.1f} minutes')
print(f'Model saved to: {MODEL_OUTPUT}/model-finetuned-final')
print('=' * 60)

## Cell 6: Test Fine-tuned Model

In [ ]:
# Load fine-tuned model
finetuned_nlp = spacy.load(f'{MODEL_OUTPUT}/model-finetuned-final')

# Test samples
test_texts = [
    'John Smith is a Senior Software Engineer at Google with 5 years of experience in Python, JavaScript, and React. Email: john.smith@gmail.com',
    'Jane Doe, Data Scientist at Microsoft. Graduated from MIT with a Master of Science in Computer Science. Located in Seattle, WA.',
    'Contact: michael.johnson@outlook.com | (555) 123-4567 | linkedin.com/in/mjohnson. AWS Certified Solutions Architect.',
]

print('=' * 60)
print('Testing Fine-tuned Model')
print('=' * 60)

for i, text in enumerate(test_texts, 1):
    print(f'\n--- Sample {i} ---')
    print(f'Text: {text[:60]}...')
    doc = finetuned_nlp(text)
    if doc.ents:
        print('Entities found:')
        for ent in doc.ents:
            print(f'  [{ent.label_}] {ent.text}')
    else:
        print('  No entities found')

## Cell 7: Export Model

In [ ]:
import shutil

# Save final model
final_model_path = f'{MODEL_OUTPUT}/resume_ner_finetuned'
finetuned_nlp.to_disk(final_model_path)

# Create zip for download
shutil.make_archive(f'{DRIVE_PATH}/resume_ner_finetuned', 'zip', final_model_path)

print('=' * 60)
print('Model Export Complete')
print('=' * 60)
print(f'\nModel saved to: {final_model_path}')
print(f'ZIP file: {DRIVE_PATH}/resume_ner_finetuned.zip')

print('\n' + '=' * 60)
print('NEXT STEPS:')
print('=' * 60)
print('1. Download resume_ner_finetuned.zip from Google Drive')
print('2. Extract to D:\\resumegpt\\backend\\models\\resume_ner\\')
print('3. Test with: python scripts/test_ner.py')